# 1. Database Import

To make these sequences more "stable", we are working locally with reference files from the latest version of reference genome from *Mus musculus* provided by ENSEMBL (version 116).

* `.gtf` file - https://ftp.ensembl.org/pub/release-116/gtf/mus_musculus/Mus_musculus.GRCm39.116.gtf.gz
* `.fasta` file - https://ftp.ensembl.org/pub/release-116/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz

To work with the reference files from ENSEMBL, we are using the `pyranges` package (https://pyranges.readthedocs.io/en/latest/index.html)

In [1]:
import pyranges as pr, pyfaidx
import pandas as pd
from IPython.display import display, Markdown
from Bio import SeqIO

# -------- Setting the Path for the necessary files -------- #

GTF_v116 = "/srv/ext_home/asbarros/genomes/mmusculus/Ensembl_v116/Mus_musculus.GRCm39.116.gtf.gz"
FA_v116  = "/srv/ext_home/asbarros/genomes/mmusculus/Ensembl_v116/Mus_musculus.GRCm39.dna.primary_assembly.fa"

# -------- Reference Genome (.fasta) -------- #
#fa  = pyfaidx.Fasta(FA_v116)                                                 # Index the fasta file (creating the .fai file) - Only requires to be done one time

In [2]:
# -------- Reference GTF -------- #
gtf = pr.read_gtf(GTF_v116, duplicate_attr=True)                              # Import the GTF file, which has all the information necessary to extract the coordinates. Keep all attributes in single string (e.g. keep tag 'Ensembl_canonical')
gtf_canonical = gtf[(gtf.Feature == "transcript") & 
                     gtf.tag.str.contains("Ensembl_canonical", na=False)]     # Filter to include only transcripts that contain the tag "Ensembl_canonical" (which corresponds to canonical transcripts)

## 2. Extract information from genes

Using the example of `Nfe2l2` and `Blvra` genes.

In [3]:
genes = ["Nfe2l2","Blvra"]                                        # List of genes of interest
gtf_sub = gtf_canonical[gtf_canonical.gene_name.isin(genes)]      # Using the subset for canonical transcripts, select the ones for the genes of interest.

print(gtf_sub.df['gene_name'].duplicated().any())                 # Check if more than one transcript was given per gene_name; if this is true - check the object

False


**Sanity Check:** Print out `gtf_sub` to confirm output. Register the Start & End coordinates

In [27]:
gtf_sub

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,2,ensembl_havana,transcript,126912566,126939017,.,+,.,ENSMUSG00000001999,17,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS16692,NaN,NaN
1,2,ensembl_havana,transcript,75504738,75534985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN


In [5]:
prom_1kb = gtf_sub.five_end().slack({"5": 1000, "3": 200})                             # 1kb promoters. Strand-Aware function, that will change the columns 'Start' & 'End' in the PyRanges object.
prom_2kb = gtf_sub.five_end().slack({"5": 2000, "3": 200})                             # 2kb promoters. Strand-Aware function, that will change the columns 'Start' & 'End' in the PyRanges object.

display(Markdown(f'**prom_1kb is a PyRanges?** {isinstance(prom_1kb, pr.PyRanges)}'))  # Confirm that the resulting object is indeed a PyRanges object
display(Markdown(f'**prom_2kb is a PyRanges?** {isinstance(prom_2kb, pr.PyRanges)}'))  # Confirm that the resulting object is indeed a PyRanges object

**prom_1kb is a PyRanges?** True

**prom_2kb is a PyRanges?** True

**Sanity Check:** Print out `prom_1kb` to confirm output. 

* The `Start` coordinates should be -1000bp from `Start` in `gtf_sub` when gene is *strand +* or +200bp when *strand is -*.

* The `End` coordinates should be +200bp from `Start` in `gtf_sub` when gene is *strand +* or -1000bp when *strand is -*.

In [6]:
prom_1kb

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,2,ensembl_havana,transcript,126911566,126912767,.,+,.,ENSMUSG00000001999,17,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS16692,NaN,NaN
1,2,ensembl_havana,transcript,75534784,75535985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN


**Sanity Check:** Print out `prom_2kb` to confirm output. 

* The `Start` coordinates should be -2000bp from `Start` in `gtf_sub` when gene is *strand +* or +200bp when *strand is -*.

* The `End` coordinates should be +200bp from `Start` in `gtf_sub` when gene is *strand +* or -2000bp when *strand is -*.

In [28]:
prom_2kb

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,ccds_id,protein_id,protein_version
0,2,ensembl_havana,transcript,126910566,126912767,.,+,.,ENSMUSG00000001999,17,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 14),NaN,NaN,NaN,CCDS16692,NaN,NaN
1,2,ensembl_havana,transcript,75534784,75536985,.,-,.,ENSMUSG00000015839,8,...,ensembl_havana,protein_coding,"CCDS,gencode_basic,gencode_primary,Ensembl_can...",1 (assigned to previous version 4),NaN,NaN,NaN,CCDS16150,NaN,NaN


# 3. Extract sequences based on the coordinates

In [8]:
seqs_1kb = pr.get_sequence(prom_1kb, path=FA_v116)                        # Based on the coordinates identified before, we get the sequences from the reference genome
seqs_2kb = pr.get_sequence(prom_2kb, path=FA_v116)                        # Based on the coordinates identified before, we get the sequences from the reference genome

display(Markdown(f'**1kb Promoter Size** {seqs_1kb.str.len().tolist()}'))
display(Markdown(f'**2kb Promoter Size** {seqs_2kb.str.len().tolist()}'))

**1kb Promoter Size** [1201, 1201]

**2kb Promoter Size** [2201, 2201]

# 4. Saving the sequences into a `fasta` file

In [9]:
df_1kb                 = seqs_1kb.to_frame(name="DNA_Sequence")
df_1kb["Gene_Name"]    = prom_1kb.df['gene_name']
df_1kb["TranscriptID"] = prom_1kb.df["transcript_id"]

In [10]:
with open("/srv/ext_home/asbarros/projects/MSoares/DNA_007/data/00a_promoters/NovelPromoters_1kb.fasta", "w") as file:
    for index, row in df_1kb.iterrows():
        # Combine your metadata into a single header line
        header = f">{row['Gene_Name']}|{row['TranscriptID']}"
        
        # Write the header and the sequence
        file.write(f"{header}\n")
        file.write(f"{row['DNA_Sequence']}\n")

In [11]:
df_2kb                 = seqs_2kb.to_frame(name="DNA_Sequence")
df_2kb["Gene_Name"]    = prom_2kb.df['gene_name']
df_2kb["TranscriptID"] = prom_2kb.df["transcript_id"]

In [12]:
with open("/srv/ext_home/asbarros/projects/MSoares/DNA_007/data/00a_promoters/NovelPromoters_2kb.fasta", "w") as file:
    for index, row in df_2kb.iterrows():
        # Combine your metadata into a single header line
        header = f">{row['Gene_Name']}|{row['TranscriptID']}"
        
        # Write the header and the sequence
        file.write(f"{header}\n")
        file.write(f"{row['DNA_Sequence']}\n")

**`Fasta` files produced!**

---

In [14]:
r = prom_2kb.df.query("gene_name == 'Blvra'").iloc[0]     # + strand, no RC to worry about
fa = pyfaidx.Fasta(FA_v116)
manual = fa[str(r.Chromosome)][r.Start:r.End].seq.upper()  # both 0-based half-open
got    = seqs_2kb[prom_2kb.df.gene_name.eq('Blvra').values].iloc[0].upper()
print(manual == got)   # True == order is preserved, you're genuinely safe

True


In [15]:
comp = str.maketrans("ACGTNacgtn","TGCANtgcan"); rc = lambda s: s.translate(comp)[::-1]
r = prom_2kb.df.query("gene_name == 'Nfe2l2'").iloc[0]      # - strand
manual = rc(fa[str(r.Chromosome)][int(r.Start):int(r.End)].seq.upper())
got    = seqs_2kb[prom_2kb.df.gene_name.eq('Nfe2l2').values].iloc[0].upper()
print(manual == got)   # True == minus-strand path is also correct

True


# 5. Confirm if the information matches with the ones made before

In [16]:
old_1kb = pyfaidx.Fasta("/srv/ext_home/asbarros/projects/MSoares/DNA_007/data/00a_promoters/DNA_007_Gene_Promoter_Sequences_1kb.fa")
old_2kb = pyfaidx.Fasta("/srv/ext_home/asbarros/projects/MSoares/DNA_007/data/00a_promoters/DNA_007_Gene_Promoter_Sequences_2kb.fa")

## 5.1. Nrf2 (Nfe2l2) Test

In [17]:
nrf2_canonical_old_1kb = str(old_1kb['Nfe2l2_ENSMUST00000102672.5'][:1200])
nrf2_canonical_old_2kb = str(old_2kb['Nfe2l2_ENSMUST00000102672.5'][:2200])

In [18]:
nrf2_canonical_new_1kb = df_1kb['DNA_Sequence'].values[1]
nrf2_canonical_new_2kb = df_2kb['DNA_Sequence'].values[1]

In [19]:
display(Markdown(f'**Nfe2l2 - 1kb Comparison**'))
display(Markdown(f'*Do the methods match?* {nrf2_canonical_old_1kb == nrf2_canonical_new_1kb}'))
display(Markdown(f'*Do the methods match (+1 extra position)?* {nrf2_canonical_old_1kb in nrf2_canonical_new_1kb}'))

**Nfe2l2 - 1kb Comparison**

*Do the methods match?* False

*Do the methods match (+1 extra position)?* True

In [20]:
display(Markdown(f'**Nfe2l2 - 2kb Comparison**'))
display(Markdown(f'*Do the methods match?* {nrf2_canonical_old_2kb == nrf2_canonical_new_2kb}'))
display(Markdown(f'*Do the methods match (+1 extra position)?* {nrf2_canonical_old_2kb in nrf2_canonical_new_2kb}'))

**Nfe2l2 - 2kb Comparison**

*Do the methods match?* False

*Do the methods match (+1 extra position)?* True

---

## 5.2. Blvra Test

In [21]:
blvra_canonical_old_1kb = str(old_1kb['Blvra_ENSMUST00000002064.15'][:1200])
blvra_canonical_old_2kb = str(old_2kb['Blvra_ENSMUST00000002064.15'][:2200])

In [22]:
blvra_canonical_new_1kb = df_1kb['DNA_Sequence'].values[0]
blvra_canonical_new_2kb = df_2kb['DNA_Sequence'].values[0]

In [23]:
display(Markdown(f'**Blvra - 1kb Comparison**'))
display(Markdown(f'*Do the methods match?* {blvra_canonical_old_1kb == blvra_canonical_new_1kb}'))
display(Markdown(f'*Do the methods match (+1 extra position)?* {blvra_canonical_old_1kb in blvra_canonical_new_1kb}'))

**Blvra - 1kb Comparison**

*Do the methods match?* False

*Do the methods match (+1 extra position)?* False

In [24]:
display(Markdown(f'**Blvra - 1kb Comparison**'))
display(Markdown(f'*Do the methods match?* {blvra_canonical_old_2kb == blvra_canonical_new_2kb}'))
display(Markdown(f'*Do the methods match (+1 extra position)?* {blvra_canonical_old_2kb in blvra_canonical_new_2kb}'))

**Blvra - 1kb Comparison**

*Do the methods match?* False

*Do the methods match (+1 extra position)?* False

In [25]:
from Bio import Align

# 1. Create an aligner object
aligner = Align.PairwiseAligner()

# Optional: Configure scoring (defaults to 1 for match, 0 for mismatch)
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -0.5
aligner.extend_gap_score = -0.1

In [26]:
# 2. Perform the alignment
alignments = aligner.align(blvra_canonical_new_2kb,blvra_canonical_old_2kb)

# 3. Print the best alignment
print(alignments[0])

target            0 TTTAAAACTTTGGACCAGTACATATTCTATCTAAACTCACTGCATTATATTCTGAATCAG
                  0 --------------------||||||||||||||||||||||||||||||||||||||||
query             0 --------------------CATATTCTATCTAAACTCACTGCATTATATTCTGAATCAG

target           60 GCATCGTCAACCTTTTTGATCACAAAATAGGGAGAATTTTAGGTCTAGTTCTTGCGTGGG
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            40 GCATCGTCAACCTTTTTGATCACAAAATAGGGAGAATTTTAGGTCTAGTTCTTGCGTGGG

target          120 GCCATCCATAACACCTAGTCTCTTTACTTTATAATCAGCCAATGAGGGTAGCGCTTAGGG
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           100 GCCATCCATAACACCTAGTCTCTTTACTTTATAATCAGCCAATGAGGGTAGCGCTTAGGG

target          180 GAAGAATGGCTGTGTGCTGTGTGCCGGGCACCATTACCAAGAATGGAGGGCAGCAGAGCC
                180 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           160 GAAGAATGGCTGTGTGCTGTGTGCCGGGCACCATTACCAAGAATGGAGGGCAGCAGAGCC

target          240 AGGC